# Lakeside heating DSM — PyTorch architectures → ONNX

**Deep-learning companion** to the sklearn walkthrough. Prefer **E+ farm** parquet;
compare **MLP / ResMLP / 1D-CNN**, peak-weighted loss, export champion to **ONNX**
for the Rust desktop walk.

| | |
|---|---|
| **Target** | `facility_kw` |
| **Peak metric** | HE 05–09 morning MAE |
| **Data** | `train_parquet_path()` → `ENERGYPLUS_SIMULATED` when farm present |
| **Artifacts** | `ml/artifacts/heating_dsm_hourly_v1.onnx` + `_feature_meta.json` |
| **Honesty** | IdealLoads+COP farm or BAS proxy · **CANDIDATE** |

CLI equivalent: `python -u ml/train_heating_dsm_torch.py`. Spec: `vibe22_agent_spec/HEATING_DSM.md`.


## 0 · Setup

In [1]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import onnxruntime as ort

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
ML = ROOT / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import artifact_paths, train_parquet_path
from feature_compile_heating_dsm import matrix_xy, morning_peak_mask, cost_from_hourly_kw
from train_heating_dsm_torch import (
    MLP, ResMLP, HourCNN, bake_off_torch, export_onnx,
)

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device, "torch", torch.__version__)


device cpu torch 2.13.0+cpu


## 1 · Load data

In [2]:
import subprocess
pq = train_parquet_path()
if not pq.is_file():
    farm_script = ROOT / "scripts" / "eplus_heating_dsm_farm.py"
    if farm_script.is_file():
        subprocess.check_call([sys.executable, "-u", str(farm_script)], cwd=str(ROOT))
        pq = train_parquet_path()
    if not pq.is_file():
        subprocess.check_call([sys.executable, "-u", str(ML / "build_bootstrap_dataset.py")], cwd=str(ROOT))
        pq = train_parquet_path()
df = pd.read_parquet(pq)
src = str(df["provenance"].iloc[0]) if "provenance" in df.columns and len(df) else "unknown"
X, y, groups, cols = matrix_xy(df)
peak = morning_peak_mask(df)
print("parquet", pq.name, "provenance", src)
print(df.shape, "n_features", len(cols))


parquet heating_dsm_eplus_farm_hourly.parquet provenance ENERGYPLUS_SIMULATED
(2880, 33) n_features 39


## 2 · Architecture bake-off

Uses GroupKFold by day and peak-weighted MSE (morning hours ×2.5).

In [3]:
# epochs=30 is a reasonable laptop default; bump for final ship
result = bake_off_torch(df, n_splits=3, epochs=30, device=device)
cv = pd.DataFrame(
    [{"family": k, **v} for k, v in result["cv"].items()]
).sort_values("mae_peak_05_09")
display(cv)
print("champion", result["champion"])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(cv["family"], cv["mae_peak_05_09"], color="#4C78A8")
ax.set_xlabel("OOF morning-peak MAE [kW]")
ax.set_title("PyTorch architecture bake-off")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.savefig(PATHS["figures"] / "torch_leaderboard.png", dpi=140, bbox_inches="tight")
plt.show()

,family,mae_peak_05_09
1,resmlp,29.414284
0,mlp,43.839326
2,hour_cnn,57.010818


champion resmlp


## 3 · Export ONNX + round-trip

In [4]:
export_onnx(result["model"], result["n_in"], PATHS["onnx"], device=device)

meta = {
    "feature_cols": result["feature_cols"],
    "scaler_mean": result["scaler"].mean_.tolist(),
    "scaler_scale": result["scaler"].scale_.tolist(),
    "champion": result["champion"],
    "cv": result["cv"],
    "schema": "lakeside.heating_dsm_hourly.v1",
    "training_parquet": str(pq),
    "training_source": src,
    "honesty": (
        f"PyTorch CANDIDATE on {src}. ONNX for desktop walk. "
        "IdealLoads+COP proxy when ENERGYPLUS_SIMULATED — not tariff-grade."
    ),
}
PATHS["feature_meta"].write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

Xs = result["scaler"].transform(X[:16])
with torch.no_grad():
    torch_pred = result["model"](torch.tensor(Xs, dtype=torch.float32)).numpy()
sess = ort.InferenceSession(str(PATHS["onnx"]), providers=["CPUExecutionProvider"])
onnx_pred = sess.run(None, {"features": Xs.astype(np.float32)})[0].reshape(-1)
max_abs = float(np.max(np.abs(torch_pred - onnx_pred)))
print("wrote", PATHS["onnx"])
print("wrote", PATHS["feature_meta"])
print("ONNX round-trip max |Δ|", max_abs)
assert max_abs < 1e-4, "ONNX mismatch"

sample = df.sort_values(["day", "hour_ending"]).groupby("simulation_id").head(24).head(24)
Xs24 = result["scaler"].transform(matrix_xy(sample)[0])
with torch.no_grad():
    kw24 = result["model"](torch.tensor(Xs24, dtype=torch.float32)).numpy()
costs = cost_from_hourly_kw(kw24, energy_rate_per_kwh=0.12, demand_rate_per_kw=15.0)
print("demo day cost stub", {k: round(v, 2) if isinstance(v, float) else v for k, v in costs.items()})
pd.DataFrame({"torch": torch_pred, "onnx": onnx_pred}).head()


wrote C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22\ml\artifacts\heating_dsm_hourly_v1.onnx
wrote C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22\ml\artifacts\heating_dsm_hourly_v1_feature_meta.json
ONNX round-trip max |Δ| 4.57763671875e-05
demo day cost stub {'energy_kwh': 2538.68, 'peak_kw': 246.09, 'energy_cost': 304.64, 'demand_cost': 3691.37, 'total_cost': 3996.01, 'annual_energy_cost_stub': 27417.74, 'annual_demand_cost_stub': 44296.45, 'annual_total_stub': 71714.18, 'energy_rate_per_kwh': 0.12, 'demand_rate_per_kw': 15.0, 'similar_days_per_year': 90.0}


,torch,onnx
0,73.454308,73.454315
1,75.736183,75.736183
2,79.849426,79.849426
3,100.907715,100.907700
4,134.907990,134.907974


## 4 · Inference sketch (desktop / sim loops)

```python
import onnxruntime as ort, json, numpy as np
meta = json.loads(open("ml/artifacts/heating_dsm_hourly_v1_feature_meta.json").read())
sess = ort.InferenceSession("ml/artifacts/heating_dsm_hourly_v1.onnx")
x = (raw_features - mean) / scale   # same order as meta["feature_cols"]
kw = sess.run(None, {"features": x.astype("float32")})[0]
```

Rust desktop (`desktop/`) loads the same ONNX + meta. Prefer farm parquet
(`ENERGYPLUS_SIMULATED`) for training — **do not change** `FEATURE_COLS` without
bumping the schema version.
